# CLIP-ReID × MPDD  드라이버 노트북

[Syliz517/CLIP-ReID](https://github.com/Syliz517/CLIP-ReID) 를 클론해서 MPDD 로 2-stage 파인튜닝.

- **stage 1**: CLIP 인코더 freeze, 개체별 텍스트 프롬프트 학습
- **stage 2**: 텍스트 freeze, image encoder 파인튜닝 (ID loss + triplet)
- 평가: query/gallery, mAP + Rank-1 (같은 pid·같은 pose 는 junk 제외)

GPU 필요. Colab 이면 런타임 → GPU. 로컬 Windows 면 아래 `NUM_WORKERS` 를 0 으로.


## 1. 레포 클론 + 의존성

In [ ]:
import os

CLIP_REID_DIR = r"C:\HyeonKyu\for_mate\ML\external\CLIP-ReID"
if not os.path.isdir(CLIP_REID_DIR):
    !git clone https://github.com/Syliz517/CLIP-ReID.git "{CLIP_REID_DIR}"
%cd {CLIP_REID_DIR}
!pip install -q yacs timm==0.9.16 scikit-image tqdm ftfy regex opencv-python
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 2. 데이터 위치

`DATA_ROOT` 아래에 이 구조가 있어야 함:

```
<DATA_ROOT>/MPDD/pytorch/train/    <id>_c<pose>s<seq>_<n>.jpg
<DATA_ROOT>/MPDD/pytorch/gallery/  ...
<DATA_ROOT>/MPDD/pytorch/query/    ...
```


In [ ]:
import os, glob

DATA_ROOT = r"C:\HyeonKyu\for_mate\ML\dataset\raw\mpdd_release"
NUM_WORKERS = 0

hit = glob.glob(os.path.join(DATA_ROOT, "**", "MPDD", "pytorch", "train"), recursive=True) or \
      glob.glob(os.path.join(DATA_ROOT, "MPDD", "pytorch", "train"))
print("train 폴더:", hit)
assert hit, f"{DATA_ROOT} 아래에서 MPDD/pytorch/train 을 못 찾음. 압축 풀고 DATA_ROOT 수정."
# MPDD 의 부모로 정규화 (train -> pytorch -> MPDD -> 부모)
DATA_ROOT = os.path.dirname(os.path.dirname(os.path.dirname(hit[0]))).replace("\\", "/")
print("DATA_ROOT =", DATA_ROOT)

## 3. MPDD dataset 클래스 작성  (`datasets/mpdd.py`)

In [9]:
%%writefile datasets/mpdd.py
# -*- coding: utf-8 -*-
"""MPDD (Multi-pose Dog Dataset) for the CLIP-ReID / TransReID codebase."""
import glob
import os.path as osp
import re

from .bases import BaseImageDataset


class MPDD(BaseImageDataset):
    """Market-1501 스타일 스플릿(train/gallery/query). 파일명: <pid>_c<cam>s<seq>_<n>.jpg"""

    dataset_dir = "MPDD/pytorch"

    def __init__(self, root="", verbose=True, pid_begin=0, **kwargs):
        super(MPDD, self).__init__()
        self.dataset_dir = osp.join(root, self.dataset_dir)
        self.train_dir = osp.join(self.dataset_dir, "train")
        self.query_dir = osp.join(self.dataset_dir, "query")
        self.gallery_dir = osp.join(self.dataset_dir, "gallery")

        self._check_before_run()
        self.pid_begin = pid_begin

        train = self._process_dir(self.train_dir, relabel=True)
        query = self._process_dir(self.query_dir, relabel=False)
        gallery = self._process_dir(self.gallery_dir, relabel=False)

        if verbose:
            print("=> MPDD loaded")
            self.print_dataset_statistics(train, query, gallery)

        self.train, self.query, self.gallery = train, query, gallery
        self.num_train_pids, self.num_train_imgs, self.num_train_cams, self.num_train_vids = \
            self.get_imagedata_info(self.train)
        self.num_query_pids, self.num_query_imgs, self.num_query_cams, self.num_query_vids = \
            self.get_imagedata_info(self.query)
        self.num_gallery_pids, self.num_gallery_imgs, self.num_gallery_cams, self.num_gallery_vids = \
            self.get_imagedata_info(self.gallery)

    def _check_before_run(self):
        for d in (self.dataset_dir, self.train_dir, self.query_dir, self.gallery_dir):
            if not osp.exists(d):
                raise RuntimeError(f"'{d}' 경로가 없습니다.")

    def _process_dir(self, dir_path, relabel=False):
        img_paths = sorted(glob.glob(osp.join(dir_path, "*.jpg")))
        pattern = re.compile(r"([-\d]+)_c(\d+)")

        pid_container = set()
        for img_path in img_paths:
            m = pattern.search(osp.basename(img_path))
            if m and int(m.group(1)) != -1:
                pid_container.add(int(m.group(1)))
        pid2label = {pid: label for label, pid in enumerate(sorted(pid_container))}

        dataset = []
        for img_path in img_paths:
            m = pattern.search(osp.basename(img_path))
            if m is None:
                continue
            pid, camid = int(m.group(1)), int(m.group(2))
            if pid == -1:
                continue
            camid -= 1
            if relabel:
                pid = pid2label[pid]
            dataset.append((img_path, self.pid_begin + pid, camid, 1))
        return dataset


Overwriting datasets/mpdd.py


## 4. dataset 팩토리에 `mpdd` 등록

레포 버전에 따라 `datasets/__init__.py` 또는 `datasets/make_dataloader.py` 에 `__factory` 가 있다. 있는 쪽 전부에 주입.

In [14]:
import re, pathlib

targets = [p for p in ["datasets/__init__.py",
                       "datasets/make_dataloader.py",
                       "datasets/make_dataloader_clipreid.py"]
           if pathlib.Path(p).exists() and "__factory" in pathlib.Path(p).read_text()]
assert targets, "__factory 를 가진 파일을 못 찾음 - datasets/ 를 직접 확인"

for p in targets:
    txt = pathlib.Path(p).read_text()
    if "'mpdd'" in txt or '"mpdd"' in txt:
        print(p, "이미 등록됨"); continue
    # import 추가
    if "from .mpdd import MPDD" not in txt:
        txt = "from .mpdd import MPDD\n" + txt
    # __factory dict 첫 '{' 뒤에 항목 삽입
    txt = re.sub(r"(__factory\s*=\s*\{)", r"\1\n    'mpdd': MPDD,", txt, count=1)
    pathlib.Path(p).write_text(txt)
    print(p, "-> mpdd 등록 완료")


datasets/make_dataloader.py 이미 등록됨
datasets/make_dataloader_clipreid.py -> mpdd 등록 완료


## 5. config 만들기

레포의 실제 `configs/person/vit_clipreid.yml` 을 복사해서 **key 구조는 그대로** 두고 값만 수정
(yacs 는 없는 key 를 만나면 에러). 개는 세로로 길지 않으니 입력을 정사각형으로.

In [16]:

import re, pathlib, glob as _g

src = "configs/person/vit_clipreid.yml"
if not pathlib.Path(src).exists():
    cand = _g.glob("configs/**/vit_clipreid.yml", recursive=True)
    assert cand, "vit_clipreid.yml 을 못 찾음"
    src = cand[0]
dst = "configs/person/vit_clipreid_mpdd.yml"

txt = pathlib.Path(src).read_text()
txt = txt.replace("[256, 128]", "[256, 256]")                     # 개용 정사각형
txt = re.sub(r"NUM_WORKERS:\s*\d+", f"NUM_WORKERS: {NUM_WORKERS}", txt)
txt = re.sub(r"MAX_EPOCHS:\s*\d+", "MAX_EPOCHS: 60", txt)         # stage1/2 축소

# 기존 DATASETS: 이하(주석 잔재 포함) 제거하고 새 블록 추가
out, skip = [], False
for ln in txt.splitlines():
    if ln.strip().startswith("DATASETS:"):
        skip = True; continue
    if skip:
        if ln.strip() == "" or ln.lstrip().startswith("#") or ln[:1] in (" ", "\t"):
            continue
        skip = False
    out.append(ln)
txt = "\n".join(out).rstrip() + "\n"

dr = DATA_ROOT.replace("\\", "/")
txt += f'\nDATASETS:\n  NAMES: (\'mpdd\')\n  ROOT_DIR: (\'{dr}\')\n\nOUTPUT_DIR: "./logs/mpdd_clipreid"\n'

pathlib.Path(dst).write_text(txt)
print("wrote", dst, "\n" + "="*60 + "\n" + txt)

wrote configs/person/vit_clipreid_mpdd.yml 
MODEL:
  PRETRAIN_CHOICE: 'imagenet'
  METRIC_LOSS_TYPE: 'triplet'
  IF_LABELSMOOTH: 'on'
  IF_WITH_CENTER: 'no'
  NAME: 'ViT-B-16'
  STRIDE_SIZE: [16, 16]
  ID_LOSS_WEIGHT : 0.25
  TRIPLET_LOSS_WEIGHT : 1.0
  I2T_LOSS_WEIGHT : 1.0
  # SIE_CAMERA: True
  # SIE_COE : 1.0

INPUT:
  SIZE_TRAIN: [256, 256]
  SIZE_TEST: [256, 256]
  PROB: 0.5 # random horizontal flip
  RE_PROB: 0.5 # random erasing
  PADDING: 10
  PIXEL_MEAN: [0.5, 0.5, 0.5]
  PIXEL_STD: [0.5, 0.5, 0.5]

DATALOADER:
  SAMPLER: 'softmax_triplet'
  NUM_INSTANCE: 4
  NUM_WORKERS: 0

SOLVER:
  STAGE1:
    IMS_PER_BATCH: 64
    OPTIMIZER_NAME: "Adam"
    BASE_LR: 0.00035
    WARMUP_LR_INIT: 0.00001
    LR_MIN: 1e-6
    WARMUP_METHOD: 'linear'
    WEIGHT_DECAY:  1e-4
    WEIGHT_DECAY_BIAS: 1e-4
    MAX_EPOCHS: 60
    CHECKPOINT_PERIOD: 120
    LOG_PERIOD: 50
    WARMUP_EPOCHS: 5
  
  STAGE2:
    IMS_PER_BATCH: 64
    OPTIMIZER_NAME: "Adam"
    BASE_LR: 0.000005
    WARMUP_METHOD: 'linea

## 6. 학습  (stage 1 + stage 2 연속)

In [17]:
!python train_clipreid.py --config_file configs/person/vit_clipreid_mpdd.yml


2026-09-04 19:47:57,527 transreid INFO: Saving model in the path :./logs/mpdd_clipreid
2026-09-04 19:47:57,527 transreid INFO: Namespace(config_file='configs/person/vit_clipreid_mpdd.yml', opts=[], local_rank=0)
2026-09-04 19:47:57,528 transreid INFO: Loaded configuration file configs/person/vit_clipreid_mpdd.yml
2026-09-04 19:47:57,528 transreid INFO: 
MODEL:
  PRETRAIN_CHOICE: 'imagenet'
  METRIC_LOSS_TYPE: 'triplet'
  IF_LABELSMOOTH: 'on'
  IF_WITH_CENTER: 'no'
  NAME: 'ViT-B-16'
  STRIDE_SIZE: [16, 16]
  ID_LOSS_WEIGHT : 0.25
  TRIPLET_LOSS_WEIGHT : 1.0
  I2T_LOSS_WEIGHT : 1.0
  # SIE_CAMERA: True
  # SIE_COE : 1.0

INPUT:
  SIZE_TRAIN: [256, 256]
  SIZE_TEST: [256, 256]
  PROB: 0.5 # random horizontal flip
  RE_PROB: 0.5 # random erasing
  PADDING: 10
  PIXEL_MEAN: [0.5, 0.5, 0.5]
  PIXEL_STD: [0.5, 0.5, 0.5]

DATALOADER:
  SAMPLER: 'softmax_triplet'
  NUM_INSTANCE: 4
  NUM_WORKERS: 0

SOLVER:
  STAGE1:
    IMS_PER_BATCH: 64
    OPTIMIZER_NAME: "Adam"
    BASE_LR: 0.00035
    WARM


  0%|                                               | 0.00/351M [00:00<?, ?iB/s]
  3%|█▏                                     | 10.8M/351M [00:00<00:03, 108MiB/s]
  6%|██▍                                    | 22.5M/351M [00:00<00:02, 113MiB/s]
 10%|███▊                                   | 34.1M/351M [00:00<00:02, 115MiB/s]
 13%|█████                                  | 45.8M/351M [00:00<00:02, 115MiB/s]
 16%|██████▍                                | 57.5M/351M [00:00<00:02, 116MiB/s]
 20%|███████▋                               | 69.1M/351M [00:00<00:02, 116MiB/s]
 23%|████████▉                              | 80.8M/351M [00:00<00:02, 116MiB/s]
 26%|██████████▎                            | 92.4M/351M [00:00<00:02, 116MiB/s]
 30%|███████████▊                            | 104M/351M [00:00<00:02, 116MiB/s]
 33%|█████████████▏                          | 116M/351M [00:01<00:02, 116MiB/s]
 36%|██████████████▌                         | 127M/351M [00:01<00:01, 117MiB/s]
 40%|███████████████▊      

## 7. 평가 (best 체크포인트)

In [18]:
import glob
ckpts = sorted(glob.glob("logs/mpdd_clipreid/*.pth"))
print("체크포인트:", ckpts)
best = [c for c in ckpts if "best" in c] or ckpts[-1:]
assert best, "logs/mpdd_clipreid/ 에 .pth 가 없음 (학습이 끝났는지 확인)"
WEIGHT = best[-1] if isinstance(best, list) else best
print("평가 대상:", WEIGHT)

!python test_clipreid.py --config_file configs/person/vit_clipreid_mpdd.yml TEST.WEIGHT "$WEIGHT"


체크포인트: ['logs/mpdd_clipreid\\ViT-B-16_60.pth']
평가 대상: logs/mpdd_clipreid\ViT-B-16_60.pth
2026-09-04 19:59:16,291 transreid INFO: Namespace(config_file='configs/person/vit_clipreid_mpdd.yml', opts=['TEST.WEIGHT', 'logs/mpdd_clipreid\\ViT-B-16_60.pth'])
2026-09-04 19:59:16,291 transreid INFO: Loaded configuration file configs/person/vit_clipreid_mpdd.yml
2026-09-04 19:59:16,291 transreid INFO: 
MODEL:
  PRETRAIN_CHOICE: 'imagenet'
  METRIC_LOSS_TYPE: 'triplet'
  IF_LABELSMOOTH: 'on'
  IF_WITH_CENTER: 'no'
  NAME: 'ViT-B-16'
  STRIDE_SIZE: [16, 16]
  ID_LOSS_WEIGHT : 0.25
  TRIPLET_LOSS_WEIGHT : 1.0
  I2T_LOSS_WEIGHT : 1.0
  # SIE_CAMERA: True
  # SIE_COE : 1.0

INPUT:
  SIZE_TRAIN: [256, 256]
  SIZE_TEST: [256, 256]
  PROB: 0.5 # random horizontal flip
  RE_PROB: 0.5 # random erasing
  PADDING: 10
  PIXEL_MEAN: [0.5, 0.5, 0.5]
  PIXEL_STD: [0.5, 0.5, 0.5]

DATALOADER:
  SAMPLER: 'softmax_triplet'
  NUM_INSTANCE: 4
  NUM_WORKERS: 0

SOLVER:
  STAGE1:
    IMS_PER_BATCH: 64
    OPTIMIZER_NA

c:\HyeonKyu\for_mate\ML\clip_reid\CLIP-ReID\utils\metrics.py:12: UserWarning: This overload of addmm_ is deprecated:
	addmm_(Number beta, Number alpha, Tensor mat1, Tensor mat2)
Consider using one of the following signatures instead:
	addmm_(Tensor mat1, Tensor mat2, *, Number beta = 1, Number alpha = 1) (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\python_arg_parser.cpp:1841.)
  dist_mat.addmm_(1, -2, qf, gf.t())


---
### 참고

- **입력 종횡비**: `[256, 256]` 으로 바꿔뒀음(개용). 사람 기본값 `[256, 128]` 로 되돌리려면 셀 5 수정.
- **로컬 Windows**: 셀 2 에서 `NUM_WORKERS = 0`.
- **CLIP 가중치**: 레포 `make_model` 이 ViT-B-16 을 자동 다운로드/캐시. 안 되면 config 의 `MODEL.PRETRAIN_PATH` 에 경로 지정.
- **MegaDescriptor 비교**: 여기 mAP/Rank-1 을 `finetune_megadescriptor_official.ipynb` 결과와 **같은 MPDD query/gallery** 기준으로 나란히 놓으면 됨.
- **CARE 로 확장**: 이 stage 2 에 (1) 프롬프트를 이미지 조건부로 만드는 2층 MLP(VDTDG), (2) 개체별 텍스트 프로토타입 mean-pool + `L_i2pCon` 추가.
